In [12]:
from src.models.models import ChunkMetadata
from src.utils.document_type import DocumentType

document = ChunkMetadata(
    document_id="doc-001",
    source="docs/guide.pdf",
    chunk_index=3,
    document_type=DocumentType.PDF,
    title="Technical Guide",
)

print(document.model_dump())

{'document_id': 'doc-001', 'source': 'docs/guide.pdf', 'chunk_index': 3, 'page': None, 'document_type': <DocumentType.PDF: 'pdf'>, 'title': 'Technical Guide'}


In [17]:
from pathlib import Path
from unstructured.partition.pdf import partition_pdf
from unstructured.partition.html import partition_html
from unstructured.partition.md import partition_md


def extract_document(file_path: str) -> str:
    path = Path(file_path)

    if path.suffix.lower() == ".pdf":
        elements = partition_pdf(
            filename=str(path),
            strategy="fast",
        )
    elif path.suffix.lower() in {".html", ".htm"}:
        elements = partition_html(filename=str(path))
    elif path.suffix.lower() == ".md":
        elements = partition_md(filename=str(path))
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")

    return "\n\n".join(str(element) for element in elements)


if __name__ == "__main__":
    file_path = Path("corpus/product-docs/admin/audit-log.md")
    text = extract_document(file_path)
    print(text)

FileNotFoundError: [Errno 2] No such file or directory: 'corpus/product-docs/admin/audit-log.md'

In [18]:
from pathlib import Path

file_path = Path("corpus/product-docs/admin/audit-log.md")

print("File exists:", file_path.exists())

text = extract_document(str(file_path))
print(text)


File exists: False


FileNotFoundError: [Errno 2] No such file or directory: 'corpus/product-docs/admin/audit-log.md'

In [19]:
import os

print(os.getcwd())

/home/lspj/checkpoint-rag/notebooks


In [37]:

from pathlib import Path

from unstructured.partition.pdf import partition_pdf
from unstructured.partition.html import partition_html
from unstructured.partition.md import partition_md


def extract_document(file_path: str | Path) -> dict:
    path = Path(file_path)

    if path.suffix.lower() == ".pdf":
        elements = partition_pdf(
            filename=str(path),
            strategy="fast",
        )
    elif path.suffix.lower() in {".html", ".htm"}:
        elements = partition_html(filename=str(path))
    elif path.suffix.lower() == ".md":
        elements = partition_md(filename=str(path))
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")

    text = "\n\n".join(str(element) for element in elements)

    metadata = [
        element.metadata.to_dict()
        for element in elements
    ]

    return {
        "text": text,
        "metadata": metadata,
    }


if __name__ == "__main__":
    file_path = Path("../corpus/product-docs/admin/audit-log.md")

    result = extract_document(file_path)

    text = result["text"]
    metadata = result["metadata"]

    print("Text:", text)
    print("Metadata:", metadata)


Text: Audit Log

Track every administrative action in your workspace. Available on the Enterprise plan.

What is logged

User invitations, role changes, removals

Workflow create/update/delete

Integration connections, disconnections

API key creation and deletion

Settings changes (SSO config, IP allowlist, etc.)

Billing events

Workflow runs and integration data flow are not in the audit log; see Run logs for those.

Accessing the log

Settings → Audit Log. Each entry includes:

Timestamp (UTC)

Actor (user or API key)

Action (e.g., workflow.update)

Target (the resource affected)

IP address and user agent

Searching and filtering

Filter by date range, actor, action type, or target. Free-text search matches across all fields.

Retention

Audit log entries are retained for 365 days.

Export

Export the audit log to CSV or JSON for compliance or external SIEM ingestion. Exports are generated asynchronously and emailed when ready.

SIEM streaming

For real-time SIEM integration, con

In [1]:
from pathlib import Path

from langchain_text_splitters import MarkdownTextSplitter
from unstructured.documents.elements import Element, Title
from unstructured.partition.md import partition_md

from src.models.models import Chunk, ChunkMetadata
from src.utils.document_type import DocumentType


def load_markdown(file_path: str | Path) -> list[Element]:
    path = Path(file_path)
    if path.suffix.lower() != ".md":
        raise ValueError(f"Unsupported file type: {path.suffix}")
    return partition_md(filename=str(path))


def elements_to_text(elements: list[Element]) -> str:
    parts = []
    for el in elements:
        if isinstance(el, Title):
            depth = (el.metadata.category_depth or 0) + 1
            parts.append(f"{'#' * depth} {el.text}")  # restore heading markers
        else:
            parts.append(el.text)
    return "\n\n".join(parts)


def chunking(elements: list[Element], file_path: str | Path) -> list[Chunk]:
    path = Path(file_path)
    document_id = path.stem
    text = elements_to_text(elements)

    splitter = MarkdownTextSplitter(chunk_size=2000)
    pieces = splitter.split_text(text)

    title = next((el.text for el in elements if isinstance(el, Title)), None)

    return [
        Chunk(
            id=f"{document_id}-{i}",
            text=piece,
            metadata=ChunkMetadata(
                document_id=document_id,
                source=str(path),
                chunk_index=i,
                page=None,
                document_type=DocumentType.MARKDOWN,  # adjust to your enum
                title=title,
            ),
        )
        for i, piece in enumerate(pieces)
    ]

ModuleNotFoundError: No module named 'src'

In [6]:
from src.models.models import ChunkMetadata
from src.utils.document_type import DocumentType

document = ChunkMetadata(
    document_id="doc-001",
    source="docs/guide.pdf",
    chunk_index=3,
    document_type=DocumentType.PDF,
    title="Technical Guide",
)

print(document.model_dump())

ModuleNotFoundError: No module named 'src'

In [8]:
from pathlib import Path
from langchain_text_splitters import MarkdownTextSplitter
from unstructured.documents.elements import Element, Title
from unstructured.partition.md import partition_md
from src.models.models import Chunk, ChunkMetadata
from src.utils.document_type import DocumentType


def load_markdown(file_path: str | Path) -> list[Element]:
    path = Path(file_path)

    if path.suffix.lower() != ".md":
        raise ValueError(f"Unsupported file type: {path.suffix}")
    return partition_md(filename=str(path))


def chunking(
    elements: list[Element], file_path: str | Path, chunk_size: int = 2000
) -> list[Chunk]:
    path = Path(file_path)
    text = "\n\n".join(element.text for element in elements)
    title = next((e.text for e in elements if isinstance(e, Title)), None)

    splitter = MarkdownTextSplitter(chunk_size=chunk_size)
    document_id = path.stem

    return [
        Chunk(
            id=f"{document_id}-{index}",
            text=chunk_text,
            metadata=ChunkMetadata(
                document_id=document_id,
                source=str(path),
                chunk_index=index,
                document_type=DocumentType.MARKDOWN,
                title=title,
            ),
        )
        for index, chunk_text in enumerate(splitter.split_text(text))
    ]


if __name__ == "__main__":
    file_path = Path("corpus/product-docs/admin/audit-log.md")
    elements = load_markdown(file_path)
    for chunk in chunking(elements, file_path):
        print(chunk.model_dump_json(indent=2))

ModuleNotFoundError: No module named 'src'